# 07 · Month-over-Month Sales Trend

**Goal:** see how monthly sales move over time, with month-over-month (MoM) growth and 3- and 6-month rolling averages.

**Source:** `gold.fact_sales` (orders with a valid date)

**Output:** `monthly_sales_trend.csv` and a trend chart in `output/`

## 1. Setup

In [9]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import config
import matplotlib.pyplot as plt

os.makedirs("output", exist_ok=True)
engine = create_engine(config.get_conn_url())

## 2. Load sales

In [5]:
sales = pd.read_sql(
    """
    SELECT order_number, order_date, customer_key, product_key, sales_amount, quantity
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
    """,
    engine,
)

sales["order_date"] = pd.to_datetime(sales["order_date"])

## 3. Monthly sales

In [6]:
monthly = (
    sales.assign(order_month=sales["order_date"].dt.to_period("M"))
    .groupby("order_month")["sales_amount"]
    .sum()
    .sort_index()
)

## 4. MoM growth and rolling averages

In [7]:
trend = monthly.to_frame("sales")
trend["mom_growth_%"] = trend["sales"].pct_change() * 100
trend["rolling_3m_sales"] = trend["sales"].rolling(3).mean()
trend["rolling_6m_sales"] = trend["sales"].rolling(6).mean()

print(trend.tail(12).round(2))

               sales  mom_growth_%  rolling_3m_sales  rolling_6m_sales
order_month                                                           
2013-02       771218        -10.09         751143.33         635437.00
2013-03      1049732         36.11         892902.67         729367.50
2013-04      1045860         -0.37         955603.33         814490.00
2013-05      1284456         22.81        1126682.67         938913.00
2013-06      1642948         27.91        1324421.33        1108662.00
2013-07      1371595        -16.52        1432999.67        1194301.50
2013-08      1545910         12.71        1520151.00        1323416.83
2013-09      1447324         -6.38        1454943.00        1389682.17
2013-10      1673261         15.61        1555498.33        1494249.00
2013-11      1780688          6.42        1633757.67        1576954.33
2013-12      1874128          5.25        1776025.67        1615484.33
2014-01        45642        -97.56        1233486.00        1394492.17


The last month (2014-01) shows −97.6% MoM, with sales of 45,642 against 1,874,128 in 2013-12. The data ends on 2014-01-28 with very few January orders, so this looks like a partial month and not a real collapse. The first month (2010-12, sales 43,419) is partial in the same way. Exclude both when reading growth rates.

## 5. Trend chart

In [10]:
plt.figure(figsize=(12, 5))
plt.plot(trend.index.astype(str), trend["sales"], label="Monthly Sales")
plt.plot(trend.index.astype(str), trend["rolling_3m_sales"], label="3M Rolling Avg")
plt.xticks(rotation=70)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.savefig("output/14_monthly_sales_trend.png", dpi=120)
plt.close()

![Monthly sales trend](output/14_monthly_sales_trend.png)

## 6. Save outputs

In [11]:
trend.to_csv("output/monthly_sales_trend.csv")
print("Trend outputs saved.")

Forecast outputs saved.
